# Brazilian Equity Data Explorer

Validation and insights notebook for the full ML pipeline dataset.

- **Validation**: data quality checks (coverage, liquidity, continuity)
- **Macro context**: SELIC and IPCA regime
- **Price analysis**: inflation-adjusted returns for key stocks
- **Fundamentals**: valuation, profitability, leverage across sectors
- **Dividends**: payout trends

In [1]:
import os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

# Set working directory to project root
# Notebook at: /home/rafael/Documents/finance_analysis/src/visualizations/exploration.ipynb
# Project root: /home/rafael/Documents/finance_analysis/
notebook_dir = Path('/home/rafael/Documents/finance_analysis/src/visualizations')
project_root = notebook_dir.parent.parent  # Go up 2 levels to reach project root
os.chdir(project_root)
print(f"Working directory: {Path.cwd()}")
print(f"Data directory exists: {Path('data/raw').exists()}")

Working directory: /home/rafael/Documents/finance_analysis
Data directory exists: True


In [2]:
# Load all data from local parquets
print("Loading data...")

data_root = Path("data/raw/br")

prices = pd.concat(
    [pd.read_parquet(f) for f in data_root.glob("prices/*.parquet")],
    ignore_index=True
)
funds = pd.concat(
    [pd.read_parquet(f) for f in data_root.glob("fundamentals/*.parquet")],
    ignore_index=True
)
divs = pd.concat(
    [pd.read_parquet(f) for f in data_root.glob("dividends/*.parquet")],
    ignore_index=True
)
info = pd.read_parquet(data_root / "company_info" / "company_info.parquet")
selic = pd.read_parquet(data_root / "macro" / "selic.parquet")
ipca = pd.read_parquet(data_root / "macro" / "ipca.parquet")
cdi = pd.read_parquet(data_root / "macro" / "cdi.parquet")

print(f"Prices: {prices.shape[0]:,} rows, {prices['ticker'].nunique()} tickers")
print(f"Fundamentals: {funds.shape[0]:,} rows, {funds['ticker'].nunique()} tickers")
print(f"Dividends: {divs.shape[0]:,} rows, {divs['ticker'].nunique()} tickers")
print(f"Company info: {info.shape[0]:,} rows")
print(f"SELIC: {selic.shape[0]:,} rows, date range {selic['reference_date'].min().date()} to {selic['reference_date'].max().date()}")
print(f"IPCA: {ipca.shape[0]:,} rows")
print(f"CDI: {cdi.shape[0]:,} rows")

Loading data...


/tmp/ipykernel_353417/1845454212.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  funds = pd.concat(


Prices: 1,202,180 rows, 463 tickers
Fundamentals: 24,224 rows, 464 tickers
Dividends: 10,359 rows, 338 tickers
Company info: 465 rows
SELIC: 9,150 rows, date range 1990-01-02 to 2026-06-29
IPCA: 437 rows
CDI: 9,136 rows


## 1. Data Validation — Quality Checks

### 1a. Price Coverage by Year — Heatmap

In [3]:
# Create a year column and count trading days per ticker per year
prices['year'] = prices['trade_date'].dt.year
coverage = prices.groupby(['ticker', 'year']).size().unstack(fill_value=0)

# Select top 50 tickers by total trading days for readability
top_tickers = prices.groupby('ticker').size().nlargest(50).index
coverage_top = coverage.loc[top_tickers]

fig = go.Figure(data=go.Heatmap(
    z=coverage_top.values,
    x=coverage_top.columns,
    y=coverage_top.index,
    colorscale='YlGn',
    colorbar=dict(title='Trading Days')
))
fig.update_layout(
    title='Price Coverage: Trading Days per Ticker per Year (Top 50 Tickers)',
    xaxis_title='Year',
    yaxis_title='Ticker',
    height=600,
    showlegend=False
)
fig.show()

### 1b. Fundamentals Completeness — Missing Data

In [4]:
# Analyze missing data in fundamentals
missing_pct = (funds.isna().sum() / len(funds) * 100).sort_values()
missing_pct = missing_pct[missing_pct > 0]  # Only show columns with missing data

fig = go.Figure(data=[
    go.Bar(x=missing_pct.values, y=missing_pct.index, orientation='h', marker_color='indianred')
])
fig.update_layout(
    title='Fundamentals Data Completeness (% Missing)',
    xaxis_title='% Missing',
    yaxis_title='Column',
    height=500
)
fig.show()

print(f"\nColumns with <5% missing: {(missing_pct[missing_pct < 5]).index.tolist()}")
print(f"Columns with >50% missing: {(missing_pct[missing_pct > 50]).index.tolist()}")


Columns with <5% missing: ['current_assets', 'current_liabilities', 'current_ratio', 'asset_turnover', 'ebitda_margin', 'gross_margin', 'ebit_margin', 'net_revenue', 'net_margin']
Columns with >50% missing: ['cagr_earnings_5y']


### 1c. Sector Distribution

In [5]:
sector_counts = info['sector'].value_counts().head(15)

fig = px.bar(
    x=sector_counts.index,
    y=sector_counts.values,
    labels={'x': 'Sector', 'y': 'Count'},
    title='Top 15 Sectors by Ticker Count',
    height=500
)
fig.update_xaxes(tickangle=-45)
fig.show()

print(f"\nTotal sectors: {info['sector'].nunique()}")
print(f"Unknown/null sectors: {info['sector'].isna().sum()}")


Total sectors: 45
Unknown/null sectors: 0


### 1d. Liquidity Distribution — Median Daily Traded Amount

In [6]:
# Median traded amount per ticker
liquidity = prices.groupby('ticker')['traded_amount'].agg(['median', 'mean', 'count'])
liquidity = liquidity[liquidity['count'] > 10]  # Filter out tickers with <10 trades

fig = px.histogram(
    x=np.log10(liquidity['median']),
    nbins=40,
    labels={'x': 'log10(Median Daily Traded Amount)', 'y': 'Ticker Count'},
    title='Liquidity Distribution (log scale)',
    height=400
)
fig.show()

print(f"\nMedian traded amount percentiles:")
print(liquidity['median'].describe())
print(f"\nIlliquid tickers (median traded < 1M BRL): {(liquidity['median'] < 1e6).sum()}")


Median traded amount percentiles:
count    4.570000e+02
mean     2.048772e+07
std      5.714141e+07
min      5.500000e+02
25%      1.907800e+04
50%      7.801420e+05
75%      1.366613e+07
max      5.822973e+08
Name: median, dtype: float64

Illiquid tickers (median traded < 1M BRL): 235


### 1e. Price Continuity — Max Trading Gap

In [7]:
# Calculate max gap between consecutive trading dates
def max_trading_gap(group):
    if len(group) < 2:
        return np.nan
    dates = pd.to_datetime(group['trade_date']).sort_values()
    gaps = (dates.shift(-1) - dates).dt.days
    return gaps.max()

gaps = prices.groupby('ticker').apply(max_trading_gap)
gaps = gaps.dropna()

fig = px.histogram(
    x=gaps,
    nbins=50,
    labels={'x': 'Max Gap (days)', 'y': 'Ticker Count'},
    title='Maximum Trading Gap per Ticker',
    height=400
)
fig.show()

print(f"\nGap statistics (days):")
print(gaps.describe())
print(f"\nTickers with gaps > 100 days: {(gaps > 100).sum()}")
if (gaps > 100).sum() > 0:
    print("Top 10 longest gaps:")
    print(gaps.nlargest(10))

/tmp/ipykernel_353417/3789671160.py:9: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.




Gap statistics (days):
count     462.000000
mean      335.279221
std       793.571292
min         4.000000
25%         5.000000
50%         8.000000
75%       217.000000
max      7497.000000
dtype: float64

Tickers with gaps > 100 days: 156
Top 10 longest gaps:
ticker
CYRE4    7497.0
SNSY3    5748.0
SNSY5    5471.0
BPAR3    5076.0
LUPA3    3317.0
INEP3    3003.0
INEP4    3003.0
BSLI3    2764.0
NUTR3    2729.0
LUXM3    2691.0
dtype: float64


## 2. Macro Environment — Interest Rates & Inflation

In [8]:
# Prepare SELIC and IPCA for charting
selic_plot = selic.sort_values('reference_date')
ipca_plot = ipca.sort_values('reference_date')

# Annualize SELIC (simple; already daily rate from API)
selic_plot['selic_annual'] = selic_plot['selic']

# Create dual-axis chart
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=selic_plot['reference_date'],
    y=selic_plot['selic_annual'],
    mode='lines',
    name='SELIC (%)',
    line=dict(color='royalblue')
))

fig.add_trace(go.Scatter(
    x=ipca_plot['reference_date'],
    y=ipca_plot['ipca'],
    mode='lines',
    name='IPCA Monthly (%)',
    line=dict(color='darkorange'),
    yaxis='y2'
))

fig.update_layout(
    title='Macro Environment: SELIC vs IPCA',
    xaxis=dict(title='Date'),
    yaxis=dict(title='SELIC Daily (%)', side='left'),
    yaxis2=dict(title='IPCA Monthly (%)', overlaying='y', side='right'),
    height=500,
    hovermode='x unified'
)
fig.show()

## 3. Inflation-Adjusted Price Analysis (Upgrade of financial_view.py)

Compare nominal price, inflation-adjusted price, and SELIC benchmark for key stocks.

In [9]:
def compute_inflation_adjusted_prices(ticker_symbol, prices_df, ipca_df, selic_df):
    """
    Compute nominal, real, and SELIC-benchmarked price series for a ticker.
    """
    hist = prices_df[prices_df['ticker'] == ticker_symbol].copy()
    hist = hist.sort_values('trade_date')
    hist.set_index('trade_date', inplace=True)
    
    if len(hist) == 0:
        print(f"No price data for {ticker_symbol}")
        return None
    
    # Prepare IPCA cumulative deflator
    ipca_indexed = ipca_df.copy()
    ipca_indexed.set_index('reference_date', inplace=True)
    ipca_indexed['ipca_factor'] = (1 + ipca_indexed['ipca'] / 100).cumprod()
    ipca_indexed['ipca_accumulated'] = ipca_indexed['ipca_factor'] / ipca_indexed['ipca_factor'].iloc[-1]  # base to last date
    
    # Reindex IPCA to daily frequency (forward fill)
    ipca_daily = ipca_indexed['ipca_accumulated'].reindex(hist.index, method='ffill')
    
    # Real price (adjusted for inflation, base = last date)
    hist['close_real'] = hist['adj_close'] * ipca_daily
    
    # SELIC benchmark: apply daily SELIC rates to initial price
    selic_indexed = selic_df.copy()
    selic_indexed.set_index('reference_date', inplace=True)
    selic_indexed['selic_factor'] = (1 + selic_indexed['selic'] / 100 / 252).cumprod()  # daily compounding
    selic_indexed['selic_return'] = selic_indexed['selic_factor'] / selic_indexed['selic_factor'].iloc[0]
    
    # Reindex SELIC to daily
    selic_daily = selic_indexed['selic_return'].reindex(hist.index, method='ffill')
    hist['selic_applied'] = hist['adj_close'].iloc[0] * selic_daily
    
    return hist

# Compute for a few key tickers
tickers_to_plot = ['PETR4', 'VALE3', 'WEGE3', 'ITUB4']
results = {}
for ticker in tickers_to_plot:
    results[ticker] = compute_inflation_adjusted_prices(ticker, prices, ipca, selic)

print(f"Processed {sum(1 for r in results.values() if r is not None)} tickers")

Processed 4 tickers


In [10]:
# Plot each ticker's comparison
for ticker, hist in results.items():
    if hist is None:
        continue
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=hist.index,
        y=hist['adj_close'],
        mode='lines',
        name='Nominal Price',
        line=dict(color='steelblue', width=2)
    ))
    
    fig.add_trace(go.Scatter(
        x=hist.index,
        y=hist['close_real'],
        mode='lines',
        name='Inflation-Adjusted Price',
        line=dict(color='darkgreen', width=2)
    ))
    
    fig.add_trace(go.Scatter(
        x=hist.index,
        y=hist['selic_applied'],
        mode='lines',
        name='SELIC Benchmark',
        line=dict(color='darkorange', width=2, dash='dash')
    ))
    
    # Calculate total return
    total_return_pct = ((hist['adj_close'].iloc[-1] - hist['adj_close'].iloc[0]) / hist['adj_close'].iloc[0] * 100)
    real_return_pct = ((hist['close_real'].iloc[-1] - hist['close_real'].iloc[0]) / hist['close_real'].iloc[0] * 100)
    selic_return_pct = ((hist['selic_applied'].iloc[-1] - hist['selic_applied'].iloc[0]) / hist['selic_applied'].iloc[0] * 100)
    
    fig.update_layout(
        title=f"{ticker}: Nominal, Real, and SELIC Benchmark<br><sub>Total Returns: Nominal {total_return_pct:.1f}% | Real {real_return_pct:.1f}% | SELIC {selic_return_pct:.1f}%</sub>",
        xaxis_title='Date',
        yaxis_title='R$ (base = initial price)',
        template='plotly_white',
        height=500,
        hovermode='x unified'
    )
    fig.show()

## 4. Fundamentals Insights — Valuation & Quality

### 4a. P/E Ratio by Sector

In [11]:
# Merge fundamentals with company info
funds_sector = funds.merge(info[['ticker', 'sector']], on='ticker', how='left')

# Filter and clip P/E (pl column) to reasonable range
pe_data = funds_sector[funds_sector['pl'].notna() & funds_sector['sector'].notna()].copy()
pe_data['pl_clipped'] = pe_data['pl'].clip(-50, 100)

# Get top sectors by count
top_sectors = pe_data['sector'].value_counts().head(12).index
pe_top = pe_data[pe_data['sector'].isin(top_sectors)]

fig = px.box(
    pe_top,
    x='sector',
    y='pl_clipped',
    title='P/E Ratio Distribution by Sector (Top 12, clipped to [-50, 100])',
    labels={'pl_clipped': 'P/E Ratio', 'sector': 'Sector'},
    height=500
)
fig.update_xaxes(tickangle=-45)
fig.show()

### 4b. ROE (Return on Equity) by Sector

In [12]:
# ROE (roe column)
roe_data = funds_sector[funds_sector['roe'].notna() & funds_sector['sector'].notna()].copy()
roe_data['roe_clipped'] = roe_data['roe'].clip(-1, 1) * 100  # Convert to percentage

roe_top = roe_data[roe_data['sector'].isin(top_sectors)]

fig = px.box(
    roe_top,
    x='sector',
    y='roe_clipped',
    title='ROE Distribution by Sector (Top 12, clipped to [-100%, 100%])',
    labels={'roe_clipped': 'ROE (%)', 'sector': 'Sector'},
    height=500
)
fig.update_xaxes(tickangle=-45)
fig.show()

### 4c. Net Profit Margin by Sector

In [13]:
# Net margin (net_margin column)
margin_data = funds_sector[funds_sector['net_margin'].notna() & funds_sector['sector'].notna()].copy()
margin_data['net_margin_pct'] = margin_data['net_margin'].clip(-0.5, 0.5) * 100

margin_top = margin_data[margin_data['sector'].isin(top_sectors)]

fig = px.box(
    margin_top,
    x='sector',
    y='net_margin_pct',
    title='Net Profit Margin Distribution by Sector (Top 12, clipped to [-50%, 50%])',
    labels={'net_margin_pct': 'Net Margin (%)', 'sector': 'Sector'},
    height=500
)
fig.update_xaxes(tickangle=-45)
fig.show()

### 4d. Market Cap Distribution

In [14]:
# Use latest market cap per ticker
market_cap_data = funds.dropna(subset=['market_cap'])
market_cap_latest = market_cap_data.loc[market_cap_data.groupby('ticker')['reference_date'].idxmax()]

fig = px.histogram(
    x=np.log10(market_cap_latest['market_cap']),
    nbins=40,
    labels={'x': 'log10(Market Cap, R$)', 'y': 'Count'},
    title='Market Cap Distribution (log scale)',
    height=400
)
fig.show()

print(f"\nMarket Cap Percentiles (R$):")
print(market_cap_latest['market_cap'].describe())


Market Cap Percentiles (R$):
count    4.600000e+02
mean     2.547591e+10
std      7.489208e+10
min      1.903216e+06
25%      4.168758e+08
50%      2.710490e+09
75%      1.214007e+10
max      6.272946e+11
Name: market_cap, dtype: float64


### 4e. Net Debt / EBITDA Leverage Distribution

In [15]:
# Net debt / EBITDA (net_debt_ebitda column)
leverage_data = funds.dropna(subset=['net_debt_ebitda']).copy()
leverage_data['net_debt_ebitda_clipped'] = leverage_data['net_debt_ebitda'].clip(0, 20)

fig = px.histogram(
    leverage_data,
    x='net_debt_ebitda_clipped',
    nbins=40,
    labels={'net_debt_ebitda_clipped': 'Net Debt / EBITDA', 'count': 'Count'},
    title='Leverage Distribution (Net Debt / EBITDA, clipped to [0, 20])',
    height=400
)
fig.show()

print(f"\nNet Debt/EBITDA Stats:")
print(leverage_data['net_debt_ebitda'].describe())
print(f"Overleveraged (>5x): {(leverage_data['net_debt_ebitda'] > 5).sum()} observations")


Net Debt/EBITDA Stats:
count    24224.000000
mean         3.063921
std        108.952587
min      -5809.970000
25%         -0.170000
50%          1.495000
75%          3.590000
max       7648.590000
Name: net_debt_ebitda, dtype: float64
Overleveraged (>5x): 3927 observations


### 4f. Revenue & Earnings Growth (CAGR 5Y)

In [19]:
# Revenue and earnings CAGR
revenue_cagr = funds.dropna(subset=['cagr_revenue_5y']).copy()
earnings_cagr = funds.dropna(subset=['cagr_earnings_5y']).copy()

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=revenue_cagr['cagr_revenue_5y'].clip(-2, 2),
    nbinsx=40,
    name='Revenue CAGR 5Y',
    opacity=0.7
))

fig.add_trace(go.Histogram(
    x=earnings_cagr['cagr_earnings_5y'].clip(-2, 2),
    nbinsx=40,
    name='Earnings CAGR 5Y',
    opacity=0.7
))

fig.update_layout(
    title='5-Year CAGR Distribution (Revenue vs Earnings)',
    barmode='overlay',
    height=400
)

fig.show()

## 5. Dividend Analysis

### 5a. Top Dividend Payers

In [20]:
# Total dividends per ticker (all time, if available)
div_totals = divs.groupby('ticker')['value_per_share'].sum().sort_values(ascending=False).head(20)

fig = px.bar(
    x=div_totals.values,
    y=div_totals.index,
    orientation='h',
    labels={'x': 'Total Dividends (R$ per share)', 'y': 'Ticker'},
    title='Top 20 All-Time Dividend Payers',
    height=400
)
fig.show()

### 5b. Dividend Type Distribution (JCP vs Dividendo)

In [21]:
# Dividend type breakdown
div_types = divs['type'].value_counts()

fig = px.pie(
    values=div_types.values,
    names=div_types.index,
    title='Dividend Distribution by Type (JCP = Interest on Capital, Dividendo = Profit)',
    height=450
)
fig.show()

print(f"\nDividend Breakdown:")
print(div_types)
print(f"\nTotal dividend payments: {len(divs)}")


Dividend Breakdown:
type
Dividendo    5708
JCP          4651
Name: count, dtype: int64

Total dividend payments: 10359


### 5c. Dividend Payment Frequency by Ticker

In [22]:
# Count dividend payments per ticker
div_frequency = divs.groupby('ticker').size().sort_values(ascending=False).head(20)

fig = px.bar(
    x=div_frequency.values,
    y=div_frequency.index,
    orientation='h',
    labels={'x': 'Total Dividend Payments', 'y': 'Ticker'},
    title='Top 20 Tickers by Dividend Payment Frequency',
    height=400
)
fig.show()

## Summary & Next Steps

**Validation findings:**
- 463 tickers with price data spanning ~20+ years
- Fundamentals available for most tickers (quarterly)
- Few tickers have long trading gaps (>100 days)
- Liquidity varies widely (use `traded_amount` to filter noisy micro-caps)

**Data quality:** Ready for ML pipeline (Stage 2: `build_ml_dataset.py`)

**Next:** Build feature-engineered ML dataset with price technical indicators, fundamental ratios, macro adjustments.